<a href="https://colab.research.google.com/github/jl17243-commits/Advanced-Learning/blob/main/Final_project_part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers==4.40.0 "datasets<3.0.0" torch scikit-learn numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import DistilBertTokenizerFast,AutoTokenizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch
import warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))
dataset = load_dataset("takala/financial_phrasebank", "sentences_allagree", trust_remote_code=True)
print(dataset['train'][0:3])



In [ ]:
label_names=['negative','neutral','positive']
labels_all=dataset['train']['label']
unique,counts=np.unique(labels_all,return_counts=True)
total=len(labels_all)
print("\n Label Distribution：")
print(f"{'Class':<12} {'Number':>6} {'Percentage':>8}")
print("-" * 28)
for u, c in zip(unique, counts):
    print(f"{label_names[u]:<12} {c:>6}   {c/total*100:>6.1f}%")
print(f"{'total':<12} {total:>6}   100.0%")

split1=dataset["train"].train_test_split(test_size=0.2, seed=SEED)
split2=split1['train'].train_test_split(test_size=0.1, seed=SEED)
train_dataset=split2['train']
test_dataset=split1['test']
val_dataset=split2['test']

# Model 1

In [ ]:
model_name="distilbert-base-uncased"
tokenizer=DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize_fn(batch):
  return tokenizer(batch['sentence'], padding='max_length',max_length=128, truncation=True)

cols=['input_ids','attention_mask','label']
train_tok=train_dataset.map(tokenize_fn,batched=True)
val_tok=val_dataset.map(tokenize_fn,batched=True)
test_tok=test_dataset.map(tokenize_fn,batched=True)

for ds in [train_tok,val_tok,test_tok]:
  ds.set_format(type='torch',columns=cols)

In [ ]:
BATCH_SIZE=32
train_loader=DataLoader(train_tok,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_tok,batch_size=BATCH_SIZE)
test_loader=DataLoader(test_tok,batch_size=BATCH_SIZE)

In [ ]:
from transformers import AutoModelForSequenceClassification

NUM_LABELS=3
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
model_pt=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=NUM_LABELS)
model_pt.to(device)
model_pt.eval()

In [ ]:
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
def evaluate(model,loader,label_names=['negative','neutral','positive']):
  model.eval()
  all_preds,all_labels=[],[]
  with torch.no_grad():
    for batch in loader:
      input_ids=batch["input_ids"].to(device)
      attention_mask=batch["attention_mask"].to(device)
      labels=batch["label"]
      logits=model(input_ids=input_ids,attention_mask=attention_mask).logits
      preds=torch.argmax(logits,dim=1).cpu()
      all_preds.extend(preds.numpy())
      all_labels.extend(labels.numpy())
  acc=accuracy_score(all_labels,all_preds)
  f1=f1_score(all_labels,all_preds,average="weighted")
  return acc,f1,all_labels,all_preds

In [ ]:
acc,f1,all_labels,all_preds=evaluate(model_pt,test_loader)
print("="*45)
print("Model PT (base model)")
print("="*45)
print(f"Accuracy:{acc:.4f}")
print(f"F1 Score:{f1:.4f}")
print()
print(classification_report(all_labels,all_preds,target_names=label_names))


In [ ]:
cm=confusion_matrix(all_labels,all_preds)
fig,ax=plt.subplots(figsize=(5,4))
im=ax.imshow(cm,cmap="Blues")
plt.colorbar(im,ax=ax)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(label_names)
ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Model PT - Confusion Matrix")
for i in range(3):
  for j in range(3):
    ax.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
plt.tight_layout()
plt.show()

In [ ]:
results={}
results["Model PT"]={"accuracy":acc,"f1":f1}

# Model 2

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch.nn as nn
model_ft_head=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=NUM_LABELS,)

for param in model_ft_head.distilbert.parameters():
  param.requires_grad=False

total_params=sum(p.numel() for p in model_ft_head.parameters())
trainable_params=sum(p.numel() for p in model_ft_head.parameters() if p.requires_grad)
frozen_params=total_params-trainable_params

print("="*45)
print("Model PT + FT Classifier")
print("="*45)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters: {frozen_params:,}")
print()
model_ft_head.to(device)

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

EPOCHS=10
LR=2e-3

optimizer=AdamW(
  filter(lambda p:p.requires_grad,model_ft_head.parameters()),
  lr=LR,
  weight_decay=0.01
)

scheduler=ReduceLROnPlateau(optimizer,mode="max",patience=2,factor=0.5)
criterion=nn.CrossEntropyLoss()

In [ ]:
best_val_f1=0.0
best_state=None
train_losses=[]
val_f1_scores=[]

for epoch in range(EPOCHS):
  model_ft_head.train()
  running_loss=0.0

  for batch in train_loader:
    input_ids=batch["input_ids"].to(device)
    attention_mask=batch["attention_mask"].to(device)
    labels=batch["label"].to(device)

    optimizer.zero_grad()
    outputs=model_ft_head(input_ids=input_ids,attention_mask=attention_mask)
    loss=criterion(outputs.logits,labels)
    loss.backward()
    optimizer.step()

    running_loss+=loss.item()

  avg_loss=running_loss/len(train_loader)
  train_losses.append(avg_loss)

  val_acc,val_f1,_,_=evaluate(model_ft_head,val_loader)
  val_f1_scores.append(val_f1)
  scheduler.step(val_f1)

  if val_f1>best_val_f1:
    best_val_f1=val_f1
    best_state={k:v.clone() for k,v in model_ft_head.state_dict().items()}

  print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Val F1: {val_f1:.4f}"
          + (" ← best" if val_f1 == best_val_f1 else ""))

In [ ]:
model_ft_head.load_state_dict(best_state)
acc,f1,all_labels,all_preds=evaluate(model_ft_head,test_loader)
print("\n" + "=" * 45)
print("  Model PT + FT Classifier Valuation")
print("=" * 45)
print(f"  Accuracy : {acc:.4f}")
print(f"  F1 Score : {f1:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=label_names))
results["Model PT + FT Classifier"]={"accuracy":acc,"f1":f1}


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].plot(range(1,EPOCHS+1),train_losses,marker="o",color="#185FA5")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

axes[1].plot(range(1,EPOCHS+1),val_f1_scores,marker="o",color="#1D9E75")
axes[1].axhline(best_val_f1,linestyle="--",color="#E24B4A",linewidth=0.8,label=f"best={best_val_f1:.4f}")
axes[1].set_title("Validation F1 Score")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Score")
axes[1].legend()
axes[1].grid(alpha=0.3)

cm=confusion_matrix(all_labels,all_preds)
im=axes[2].imshow(cm,cmap="Blues")
plt.colorbar(im,ax=axes[2])
axes[2].set_xticks(range(3))
axes[2].set_yticks(range(3))
axes[2].set_xticklabels(label_names)
axes[2].set_yticklabels(label_names)
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("True")
axes[2].set_title("Model PT + FT Classifier - Confusion Matrix")
for i in range(3):
  for j in range(3):
    axes[2].text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
plt.suptitle("Model PT + FT Classifier",fontsize=13)
plt.tight_layout()
plt.show()


## Model 3: Custom Classifier Head — Required Explanations

### 1. How is this head different from the original?

The original classifier head in `DistilBertForSequenceClassification` is:

**Linear(768 → 768) → GELU → Dropout(0.1) → Linear(768 → 3)**

Our custom head differs in three key ways:

**Dimensionality Reduction**: Instead of keeping the hidden dimension at 768 throughout, we progressively compress it from 768 → 512. This reduces redundant feature dimensions and forces the network to extract more compact, discriminative representations.

**LayerNorm after every Linear layer**: The original head has no normalization between layers. We add `LayerNorm` after each `Linear` layer, which stabilizes the distribution of activations and is consistent with the normalization style used inside the DistilBERT body itself.

**Residual Connection**: The original head has no skip connections. We add a residual connection in the second block (`x = x + residual`), which provides a gradient highway during backpropagation and prevents the vanishing gradient problem as the head becomes deeper.

Our custom head structure:

**[CLS] token (768)
→ Linear(768 → 512) + LayerNorm(512) + GELU + Dropout(0.3)
→ Residual Block: Linear(512 → 512) + LayerNorm(512) + GELU + Dropout(0.2)
→ Linear(512 → 3)**

---

### 2. What inspired these design choices?

**Why LayerNorm (not BatchNorm)?**
DistilBERT's body uses `LayerNorm` internally throughout all 6 Transformer layers. Extending this same normalization style into the classifier head creates architectural consistency — gradients flow through a uniform normalization landscape from the output all the way back into the body. `BatchNorm` depends on batch statistics and behaves differently during training vs inference, which is undesirable in NLP settings where sequences are often processed one at a time.

**Why a residual connection?**
The residual connection was inspired directly by the Transformer architecture itself. Every layer inside DistilBERT wraps its sub-components (self-attention, FFN) with residual connections. By introducing a residual block in our classifier head, we extend this design principle beyond the body — ensuring that even if the linear transformation in layer 2 is unhelpful early in training, the original representation can still pass through unchanged.

**Why progressive dimensionality reduction (768 → 512 → 3)?**
This design was inspired by the information bottleneck principle. Gradually compressing the representation forces the network to retain only the most discriminative features for sentiment classification, rather than directly classifying from a high-dimensional space where noise may dominate.

---

### 3. Technical challenges of grafting the head onto the pre-existing body

**Challenge 1 — Interface alignment**
The DistilBERT body outputs a tensor of shape `(batch_size, seq_len, 768)`. Our custom head expects a 1D vector per sample of exactly 768 dimensions. We must correctly extract the `[CLS]` token representation using:
```python
cls_output = outputs.last_hidden_state[:, 0, :]   # shape: (batch, 768)
```
If the wrong token position is selected or the tensor is not properly sliced, the head will receive the wrong input shape and the `Linear(768 → 512)` layer will throw a dimension mismatch error.

**Challenge 2 — Bypassing the original forward logic**
`AutoModelForSequenceClassification` has its own `forward()` method that internally calls its built-in classifier head. We cannot simply swap out the head and reuse this forward method. Instead, we must use `DistilBertModel` (body only, no head) and write our own `forward()` that explicitly controls the full pipeline:

**tokenized input
→ DistilBERT body hidden states
→ [CLS] token extraction
→ custom head
→ logits**

This requires understanding the HuggingFace model API well enough to know which output attribute (`last_hidden_state`) to access, and how to correctly compose the two modules into a single `nn.Module`.

In [ ]:
import torch.nn as nn
class CustomClassifierHead(nn.Module):
  def __init__(self,input_dim=768,hidden_dim=512,num_labels=3):
    super().__init__()

    self.layer1=nn.Sequential(
        nn.Linear(input_dim,hidden_dim),
        nn.LayerNorm(hidden_dim),
        nn.GELU(),
        nn.Dropout(0.3),
    )

    self.layer2_main=nn.Sequential(
        nn.Linear(hidden_dim,hidden_dim),
        nn.LayerNorm(hidden_dim),
        nn.GELU(),
        nn.Dropout(0.2),
    )
    self.output=nn.Linear(hidden_dim,num_labels)

  def forward(self,x):
    x=self.layer1(x)
    residual=x
    x=self.layer2_main(x)
    x=x+residual
    x=self.output(x)
    return x


In [ ]:
from transformers import DistilBertModel
class DistilBertWithCustomHead(nn.Module):
  def __init__(self,model_name,num_labels=3):
    super().__init__()

    self.distilbert=DistilBertModel.from_pretrained(model_name)
    self.classifier=CustomClassifierHead(input_dim=768,hidden_dim=512,num_labels=num_labels)
    for param in self.distilbert.parameters():
      param.requires_grad=False

  def forward(self,input_ids,attention_mask):
    outputs=self.distilbert(input_ids=input_ids,attention_mask=attention_mask)
    cls_output=outputs.last_hidden_state[:,0,:]
    logits=self.classifier(cls_output)
    return logits

In [ ]:
model_own_head=DistilBertWithCustomHead(model_name=model_name,num_labels=NUM_LABELS)
model_own_head.to(device)

total_params=sum(p.numel() for p in model_own_head.parameters())
trainable_params=sum(p.numel() for p in model_own_head.parameters() if p.requires_grad)
frozen_params=total_params-trainable_params

print("=" * 45)
print("  Model 3 parameters")
print("=" * 45)
print(f"  total params    : {total_params:,}")
print(f"  trainable params  : {trainable_params:,}")
print(f"  frozen params    : {frozen_params:,}")

In [ ]:
EPOCHS_M3=10
LR_M3=2e-3

optimizer_m3=AdamW(
  filter(lambda p:p.requires_grad,model_own_head.parameters()),
  lr=LR_M3,
  weight_decay=0.01
)
scheduler_m3=ReduceLROnPlateau(optimizer_m3,mode="max",patience=2,factor=0.5)
criterion_m3=nn.CrossEntropyLoss()

In [ ]:
best_val_f1_m3=0.0
best_state_m3=None
train_losses_m3=[]
val_f1_score_m3=[]

for epoch in range(EPOCHS_M3):
  model_own_head.train()
  running_loss=0.0

  for batch in train_loader:
    input_ids=batch['input_ids'].to(device)
    attention_mask=batch['attention_mask'].to(device)
    labels=batch['label'].to(device)

    optimizer_m3.zero_grad()
    logits=model_own_head(input_ids,attention_mask)
    loss=criterion_m3(logits,labels)
    loss.backward()
    optimizer_m3.step()
    running_loss+=loss.item()

  avg_loss=running_loss/len(train_loader)
  train_losses_m3.append(avg_loss)

  model_own_head.eval()
  all_preds_val,all_labels_val=[],[]
  with torch.no_grad():
    for batch in val_loader:
      input_ids=batch["input_ids"].to(device)
      attention_mask=batch["attention_mask"].to(device)
      labels=batch["label"]
      logits=model_own_head(input_ids,attention_mask)
      preds=torch.argmax(logits,dim=-1).cpu()
      all_preds_val.extend(preds.numpy())
      all_labels_val.extend(labels.numpy())
  val_acc=accuracy_score(all_labels_val,all_preds_val)
  val_f1=f1_score(all_labels_val,all_preds_val,average="weighted")
  val_f1_score_m3.append(val_f1)
  scheduler_m3.step(val_f1)

  if val_f1 > best_val_f1_m3:
        best_val_f1_m3 = val_f1
        best_state_m3  = {k: v.clone() for k, v in model_own_head.state_dict().items()}

  print(f"Epoch {epoch+1:02d}/{EPOCHS_M3} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
        + (" ← best" if val_f1 == best_val_f1_m3 else ""))


In [ ]:

model_own_head.load_state_dict(best_state_m3)
model_own_head.eval()

all_preds_m3, all_labels_m3 = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"]
        logits = model_own_head(input_ids=input_ids, attention_mask=attention_mask)
        preds  = torch.argmax(logits, dim=-1).cpu()
        all_preds_m3.extend(preds.numpy())
        all_labels_m3.extend(labels.numpy())

acc_m3 = accuracy_score(all_labels_m3, all_preds_m3)
f1_m3  = f1_score(all_labels_m3, all_preds_m3, average="weighted")

print("\n" + "=" * 45)
print("  Model PT + Own FT Classifier Valuation")
print("=" * 45)
print(f"  Accuracy : {acc_m3:.4f}")
print(f"  F1 Score : {f1_m3:.4f}")
print()
print(classification_report(all_labels_m3, all_preds_m3, target_names=label_names))

results["Model PT + Own FT Classifier"] = {"accuracy": acc_m3, "f1": f1_m3}

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))


axes[0].plot(range(1, EPOCHS_M3+1), train_losses_m3, marker="o", color="#185FA5")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)


axes[1].plot(range(1, EPOCHS_M3+1), val_f1_score_m3, marker="o", color="#1D9E75")
axes[1].axhline(best_val_f1_m3, linestyle="--", color="#E24B4A",
                linewidth=0.8, label=f"best={best_val_f1_m3:.4f}")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1")
axes[1].legend(); axes[1].grid(alpha=0.3)


cm_m3 = confusion_matrix(all_labels_m3, all_preds_m3)
im    = axes[2].imshow(cm_m3, cmap="Blues")
plt.colorbar(im, ax=axes[2])
axes[2].set_xticks(range(3)); axes[2].set_yticks(range(3))
axes[2].set_xticklabels(label_names); axes[2].set_yticklabels(label_names)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")
axes[2].set_title("Confusion Matrix")
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, cm_m3[i, j], ha="center", va="center",
                     color="white" if cm_m3[i, j] > cm_m3.max()/2 else "black")

plt.suptitle("Model PT + Own FT Classifier", fontsize=13)
plt.tight_layout()
plt.show()

# Model 4: Pretrained model + FT Classifier head + FT base

In [ ]:
from jax._src.sharding_impls import num_addressable_indices
import copy
model_ft_full=AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS
)

model_ft_full.load_state_dict(best_state)
model_ft_full.to(device)

for param in model_ft_full.pre_classifier.parameters():
  param.requires_grad=False
for param in model_ft_full.classifier.parameters():
  param.requires_grad=False

for param in model_ft_full.distilbert.parameters():
  param.requires_grad=True

total_params=sum(p.numel() for p in model_ft_full.parameters())
trainable_params=sum(p.numel() for p in model_ft_full.parameters() if p.requires_grad)
frozen_params=total_params-trainable_params

print("=" * 45)
print("  Model 4 parameters")
print("=" * 45)
print(f"  total parameter    : {total_params:,}")
print(f"  trainable parameter  : {trainable_params:,} ")
print(f"  frozen parameter    : {frozen_params:,} ")

In [ ]:
EPOCHS_M4=5
LR_M4=2e-5

optimizer_m4=AdamW(
    filter(lambda p:p.requires_grad,model_ft_full.parameters()),
    lr=LR_M4,
    weight_decay=0.01
)
scheduler_m4=ReduceLROnPlateau(optimizer_m4,mode="max",patience=2,factor=0.5)
criterion_m4=nn.CrossEntropyLoss()

In [ ]:
best_val_f1_m4=0.0
best_state_m4=None
train_losses_m4=[]
val_f1_score_m4=[]

for epoch in range(EPOCHS_M4):
  model_ft_full.train()
  running_loss=0.0

  for batch in train_loader:
    input_ids=batch["input_ids"].to(device)
    attention_mask=batch["attention_mask"].to(device)
    labels=batch["label"].to(device)

    optimizer_m4.zero_grad()
    outputs=model_ft_full(input_ids=input_ids,attention_mask=attention_mask)
    loss=criterion_m4(outputs.logits,labels)
    loss.backward()
    optimizer_m4.step()
    running_loss+=loss.item()

  avg_loss=running_loss/len(train_loader)
  train_losses_m4.append(avg_loss)

  val_acc,val_f1,_,_=evaluate(model_ft_full,val_loader)
  val_f1_score_m4.append(val_f1)
  scheduler_m4.step(val_f1)

  if val_f1>best_val_f1_m4:
    best_val_f1_m4=val_f1
    best_state_m4={k:v.clone() for k,v in model_ft_full.state_dict().items()}

  print(f"Epoch {epoch+1:02d}/{EPOCHS_M4} | "
          f"Loss: {avg_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Val F1: {val_f1:.4f}"
          + (" ← best" if val_f1 == best_val_f1_m4 else ""))

In [ ]:
model_ft_full.load_state_dict(best_state_m4)

acc_m4,f1_m4,all_labels_m4,all_preds_m4=evaluate(model_ft_full,test_loader)
print("\n" + "=" * 45)
print("  Model PT + FT Classifier + FT Base Valuation")
print("=" * 45)
print(f"  Accuracy : {acc_m4:.4f}")
print(f"  F1 Score : {f1_m4:.4f}")
print()
print(classification_report(all_labels_m4, all_preds_m4, target_names=label_names))

results["Model PT + FT Classifier + FT Base"] = {"accuracy": acc_m4, "f1": f1_m4}

In [ ]:
# ── 6. 训练曲线 + 混淆矩阵 ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 左：训练 Loss
axes[0].plot(range(1, EPOCHS_M4+1), train_losses_m4, marker="o", color="#185FA5")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

# 中：验证 F1
axes[1].plot(range(1, EPOCHS_M4+1), val_f1_score_m4, marker="o", color="#1D9E75")
axes[1].axhline(best_val_f1_m4, linestyle="--", color="#E24B4A",
                linewidth=0.8, label=f"best={best_val_f1_m4:.4f}")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1")
axes[1].legend(); axes[1].grid(alpha=0.3)

# 右：混淆矩阵
cm_m4 = confusion_matrix(all_labels_m4, all_preds_m4)
im    = axes[2].imshow(cm_m4, cmap="Blues")
plt.colorbar(im, ax=axes[2])
axes[2].set_xticks(range(3)); axes[2].set_yticks(range(3))
axes[2].set_xticklabels(label_names); axes[2].set_yticklabels(label_names)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")
axes[2].set_title("Confusion Matrix")
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, cm_m4[i, j], ha="center", va="center",
                     color="white" if cm_m4[i, j] > cm_m4.max()/2 else "black")

plt.suptitle("Model PT + FT Classifier + FT Base", fontsize=13)
plt.tight_layout()
plt.show()

# Model 5

In [ ]:
class LoRALinear(nn.Module):
  def __init__(self,original_linear:nn.Linear,rank:int=8):
    super().__init__()
    self.original=original_linear
    d_out,d_in=original_linear.weight.shape
    self.A=nn.Parameter(torch.randn(rank,d_in)*0.01)
    self.B=nn.Parameter(torch.zeros(d_out,rank))

    for param in self.original.parameters():
      param.requires_grad=False

  def forward(self,x):
    return self.original(x)+(x@self.A.T)@self.B.T

In [ ]:
LORA_RANK=8

model_lora=AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS)
model_lora.load_state_dict(best_state)

for param in model_lora.parameters():
  param.requires_grad=False

for i,layer in enumerate(model_lora.distilbert.transformer.layer):
  layer.ffn.lin1=LoRALinear(layer.ffn.lin1,rank=LORA_RANK)
  layer.ffn.lin2=LoRALinear(layer.ffn.lin2,rank=LORA_RANK)

model_lora.to(device)

In [ ]:
RANK=8
D_MODEL=768
D_FFN=3072
N_LAYERS=6

lora_lin1=RANK*D_MODEL+D_FFN*RANK
lora_lin2=RANK*D_FFN+D_MODEL*RANK
per_layer=lora_lin1+lora_lin2
total_lora=per_layer*N_LAYERS

print("="*50)
print("Calculate the Trainable Parameters of LoRA")
print("="*50)
print(f"LoRA Lin1: {lora_lin1}")
print(f"LoRA Lin2: {lora_lin2}")
print(f"per layer:{per_layer}")
print(f"Total LoRA: {total_lora}")
print()

actual_trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"  actual number of trainable weights  : {actual_trainable:,}")
print(f"  same or not           : {'✓ same' if actual_trainable == total_lora else '✗ different'}")

In [ ]:

EPOCHS_M5 = 5
LR_M5=1e-3

optimizer_m5 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=LR_M5,
    weight_decay=0.01,
)
scheduler_m5 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_m5, mode="max", patience=1, factor=0.5
)
criterion_m5 = nn.CrossEntropyLoss()

In [ ]:
best_val_f1_m5   = 0.0
best_state_m5    = None
train_losses_m5  = []
val_f1_scores_m5 = []

for epoch in range(EPOCHS_M5):
    model_lora.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer_m5.zero_grad()
        outputs = model_lora(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion_m5(outputs.logits, labels)
        loss.backward()
        optimizer_m5.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses_m5.append(avg_loss)

    val_acc, val_f1, _, _ = evaluate(model_lora, val_loader)
    val_f1_scores_m5.append(val_f1)
    scheduler_m5.step(val_f1)

    if val_f1 > best_val_f1_m5:
        best_val_f1_m5 = val_f1
        best_state_m5  = {k: v.clone() for k, v in model_lora.state_dict().items()}

    print(f"Epoch {epoch+1:02d}/{EPOCHS_M5} | "
          f"Loss: {avg_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Val F1: {val_f1:.4f}"
          + (" ← best" if val_f1 == best_val_f1_m5 else ""))

In [ ]:
model_lora.load_state_dict(best_state_m5)

acc_m5, f1_m5, all_labels_m5, all_preds_m5 = evaluate(model_lora, test_loader)

print("\n" + "=" * 50)
print("  Model PT + FT Classifier + PEFT Base Valuation")
print("=" * 50)
print(f"  Accuracy : {acc_m5:.4f}")
print(f"  F1 Score : {f1_m5:.4f}")
print()
print(classification_report(all_labels_m5, all_preds_m5, target_names=label_names))

results["Model PT + FT Classifier + PEFT Base"] = {"accuracy": acc_m5, "f1": f1_m5}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(range(1, EPOCHS_M5+1), train_losses_m5, marker="o", color="#185FA5")
axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS_M5+1), val_f1_scores_m5, marker="o", color="#1D9E75")
axes[1].axhline(best_val_f1_m5, linestyle="--", color="#E24B4A",
                linewidth=0.8, label=f"best={best_val_f1_m5:.4f}")
axes[1].set_title("Validation F1"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1"); axes[1].legend(); axes[1].grid(alpha=0.3)

cm_m5 = confusion_matrix(all_labels_m5, all_preds_m5)
im    = axes[2].imshow(cm_m5, cmap="Blues")
plt.colorbar(im, ax=axes[2])
axes[2].set_xticks(range(3)); axes[2].set_yticks(range(3))
axes[2].set_xticklabels(label_names); axes[2].set_yticklabels(label_names)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")
axes[2].set_title("Confusion Matrix")
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, cm_m5[i, j], ha="center", va="center",
                     color="white" if cm_m5[i, j] > cm_m5.max()/2 else "black")

plt.suptitle("Model PT + FT Classifier + PEFT Base (LoRA)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print("\nCompare all models：")
print(f"{'model':<45} {'Accuracy':>10} {'F1':>10}")
print("-" * 67)
for name, m in results.items():
    print(f"{name:<45} {m['accuracy']:>10.4f} {m['f1']:>10.4f}")

## Results Analysis

### Overall Performance Summary

The results clearly demonstrate the progressive benefit of fine-tuning, with accuracy improving
from 0.34 (no training) to 0.96 (full fine-tuning) across the five models.

### Model-by-Model Analysis

**Model PT (Accuracy: 0.34, F1: 0.35)**
As expected, the pre-trained model with a randomly initialized classifier head performs near
random chance (33%), confirming that DistilBERT's general language knowledge alone is
insufficient for financial sentiment classification without any task-specific training.

**Model PT + FT Classifier (Accuracy: 0.87, F1: 0.87)**
Fine-tuning only the classifier head yields a dramatic improvement of +53%, jumping from 0.34
to 0.87. This demonstrates that the pre-trained DistilBERT body already encodes rich linguistic
features that are transferable to financial sentiment, even without any domain adaptation.

**Model PT + Own FT Classifier (Accuracy: 0.87, F1: 0.87)**
The custom classifier head (with LayerNorm, residual connections, and progressive dimensionality
reduction) achieves virtually identical performance to the original head (0.8698 vs 0.8720).
This suggests that for this task, the bottleneck is not the classifier head architecture but
rather the frozen body's inability to adapt to financial language.

**Model PT + FT Classifier + FT Base (Accuracy: 0.96, F1: 0.96)**
Full fine-tuning of the body delivers the strongest performance, with a further +9% gain over
head-only fine-tuning. This confirms that adapting the Transformer layers to financial vocabulary
and sentence structure is critical for reaching high accuracy. This is the performance ceiling
in our experiments.

**Model PT + FT Classifier + PEFT Base (Accuracy: 0.95, F1: 0.95)**
LoRA fine-tuning of only the FFN layers achieves 0.9492, just 1.1% below full fine-tuning,
while training only ~368,640 parameters compared to ~66,000,000 in Model 4 — a 99.4% reduction
in trainable parameters. This is arguably the most noteworthy result: LoRA recovers nearly all
of the gains from full fine-tuning at a fraction of the computational cost, supporting the
hypothesis that world knowledge in Transformer models is primarily stored in the FFN blocks.

### Key Takeaways

1. **Fine-tuning the body matters more than head design**: The jump from Model 2 to Model 4
   (+9%) far exceeds any difference between Model 2 and Model 3 (<0.1%), showing that body
   adaptation is the dominant factor in performance.

2. **LoRA is remarkably efficient**: Model 5 achieves 98.8% of Model 4's performance with
   less than 1% of its trainable parameters, making it the most practical choice when
   computational resources are limited.

3. **Transfer learning works**: Even with zero fine-tuning (Model 1), the pre-trained
   representations provide a foundation that, once a classifier head is trained, immediately
   achieves 87% accuracy — far above random baseline.

# Exporation
1.Using model to predict the emotion of self-created sentences

In [ ]:
def predict_sentiment(text,model,tokenizer,device):
  label_names=["negetive","neutral","positive"]
  label_emoji = ["🔴", "🟡", "🟢"]

  inputs=tokenizer(text,return_tensors="pt",truncation=True,padding=True,max_length=128)
  input_ids=inputs["input_ids"].to(device)
  attention_mask=inputs["attention_mask"].to(device)

  model.eval()
  with torch.no_grad():
    outputs=model(input_ids=input_ids,attention_mask=attention_mask)
    logits=outputs.logits
    probs=torch.softmax(logits,dim=-1).cpu()[0]
    pred=torch.argmax(probs).item()

  print("=" * 50)
  print(f"  input sentence : {text}")
  print("=" * 50)
  print(f"  predicted emotion : {label_emoji[pred]} {label_names[pred].upper()}")
  print()
  print("  probabilities：")
  for i, (name, emoji) in enumerate(zip(label_names, label_emoji)):
      bar   = "█" * int(probs[i].item() * 30)
      print(f"  {emoji} {name:<10} {probs[i].item():.4f}  {bar}")
  print("=" * 50)

  return label_names[pred], probs.numpy()


In [ ]:
sentences=[
    "The CEO resigned amid growing concerns about the company's future.",
    "The stock price rises a lot recently.",
    "The movie is great.",
    "I hate studying all the time"

]

for sentence in sentences:
  predict_sentiment(sentence,model_lora,tokenizer,device)
  print()

I was curious whether the fine-tuned sentiment model would generalize beyond financial text, so I tested it on both in-domain sentences (financial news) and out-of-sample sentences (everyday language). The results were telling: the model performs well on financial sentences, as expected, but struggles significantly with general text — even when the sentences are simple and unambiguous. This suggests the model has overfit to the style and vocabulary of financial language during fine-tuning.

# 2. Add Series Adapter
### What I am doing

In addition to the five required models, I implemented a **Series Adapter** — a
parameter-efficient fine-tuning (PEFT) method — and inserted it into the DistilBERT
body on top of Model 2 (PT + FT Classifier).

This experiment is interesting for two reasons:

**1. It provides a direct comparison between two PEFT paradigms.**
Model 5 uses LoRA, which is a **parallel** (additive) adaptation method — it adds a
low-rank bypass alongside existing weights. Series Adapters, by contrast, are a
**serial** (sequential) method — every token representation is forced to pass through
the adapter. These two approaches represent fundamentally different philosophies about
how to inject task-specific knowledge into a frozen model, and comparing their
performance on the same task reveals which paradigm is more effective for financial
sentiment classification.

**2. It tests whether domain adaptation can be achieved with extremely few parameters.**
Series Adapters use a bottleneck dimension of 64, meaning the model compresses the
768-dimensional representation down to 64 dimensions and back up again — a compression
ratio of 12×. If the adapter can recover performance close to full fine-tuning despite
this aggressive compression, it suggests that the domain shift between general English
and financial language can be captured by a surprisingly low-dimensional subspace.


In [ ]:
class SeriesAdapter(nn.Module):
  def __init__(self,hidden_dim=768,bottleneck_dim=64):
    super().__init__()
    self.down_proj=nn.Linear(hidden_dim,bottleneck_dim)
    self.act=nn.GELU()
    self.up_proj=nn.Linear(bottleneck_dim,hidden_dim)
    self.layernorm=nn.LayerNorm(hidden_dim)

    nn.init.normal_(self.down_proj.weight,std=0.01)
    nn.init.zeros_(self.down_proj.bias)
    nn.init.zeros_(self.up_proj.weight)
    nn.init.zeros_(self.up_proj.bias)

  def forward(self,x):
    residual=x
    x=self.down_proj(x)
    x=self.act(x)
    x=self.up_proj(x)
    return self.layernorm(x+residual)

In [ ]:
class TransformerLayerWithAdapter(nn.Module):
  def __init__(self,original_layer,bottleneck_dim=64):
    super().__init__()
    self.original_layer=original_layer
    self.adapter=SeriesAdapter(hidden_dim=768,bottleneck_dim=bottleneck_dim)

  def forward(self,x,attn_mask,head_mask,output_attentions):
    outputs=self.original_layer(x, attn_mask, head_mask, output_attentions)
    hidden_states=self.adapter(outputs[0])

    return (hidden_states,) + outputs[1:]

In [ ]:
BOTTLENECK_DIM=64
model_adapter=AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
)
model_adapter.load_state_dict(best_state)

for param in model_adapter.parameters():
  param.requires_grad=False

for i,layer in enumerate(model_adapter.distilbert.transformer.layer):
  model_adapter.distilbert.transformer.layer[i]=TransformerLayerWithAdapter(
      original_layer=layer,
      bottleneck_dim=BOTTLENECK_DIM,
  )

model_adapter.to(device)

In [ ]:
total_params     = sum(p.numel() for p in model_adapter.parameters())
trainable_params = sum(p.numel() for p in model_adapter.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print("=" * 50)
print("  Model 2 + Series Adapter parameters")
print("=" * 50)
print(f"  total parameters          : {total_params:,}")
print(f"  trainable parameters        : {trainable_params:,} ")
print(f"  frozen parameters          : {frozen_params:,}")

In [ ]:
EPOCHS_MA = 10
LR_MA     = 1e-3

optimizer_ma = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_adapter.parameters()),
    lr=LR_MA,
    weight_decay=0.01,
)
scheduler_ma = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ma, mode="max", patience=2, factor=0.5
)
criterion_ma = nn.CrossEntropyLoss()

In [ ]:
best_val_f1_ma   = 0.0
best_state_ma    = None
train_losses_ma  = []
val_f1_scores_ma = []

for epoch in range(EPOCHS_MA):
    model_adapter.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer_ma.zero_grad()
        outputs = model_adapter(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion_ma(outputs.logits, labels)
        loss.backward()
        optimizer_ma.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses_ma.append(avg_loss)

    val_acc, val_f1, _, _ = evaluate(model_adapter, val_loader)
    val_f1_scores_ma.append(val_f1)
    scheduler_ma.step(val_f1)

    if val_f1 > best_val_f1_ma:
        best_val_f1_ma = val_f1
        best_state_ma  = {k: v.clone() for k, v in model_adapter.state_dict().items()}

    print(f"Epoch {epoch+1:02d}/{EPOCHS_MA} | "
          f"Loss: {avg_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Val F1: {val_f1:.4f}"
          + (" ← best" if val_f1 == best_val_f1_ma else ""))

In [ ]:
model_adapter.load_state_dict(best_state_ma)
acc_ma, f1_ma, labels_ma, preds_ma = evaluate(model_adapter, test_loader)

print("\n" + "=" * 50)
print("  Model 2 + Series Adapter Valuation")
print("=" * 50)
print(f"  Accuracy : {acc_ma:.4f}")
print(f"  F1 Score : {f1_ma:.4f}")
print()
print(classification_report(labels_ma, preds_ma, target_names=label_names))

results["Model PT + FT Classifier + Series Adapter"] = {"accuracy": acc_ma, "f1": f1_ma}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(range(1, EPOCHS_MA+1), train_losses_ma, marker="o", color="#185FA5")
axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS_MA+1), val_f1_scores_ma, marker="o", color="#1D9E75")
axes[1].axhline(best_val_f1_ma, linestyle="--", color="#E24B4A",
                linewidth=0.8, label=f"best={best_val_f1_ma:.4f}")
axes[1].set_title("Validation F1"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1"); axes[1].legend(); axes[1].grid(alpha=0.3)

cm_ma = confusion_matrix(labels_ma,preds_ma)
im    = axes[2].imshow(cm_ma, cmap="Blues")
plt.colorbar(im, ax=axes[2])
axes[2].set_xticks(range(3)); axes[2].set_yticks(range(3))
axes[2].set_xticklabels(label_names); axes[2].set_yticklabels(label_names)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")
axes[2].set_title("Confusion Matrix")
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, cm_m5[i, j], ha="center", va="center",
                     color="white" if cm_m5[i, j] > cm_m5.max()/2 else "black")

plt.suptitle("Model PT + FT Classifier + Series Adapter", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print("\nCompare all models：")
print(f"{'model':<45} {'Accuracy':>10} {'F1':>10}")
print("-" * 67)
for name, m in results.items():
    print(f"{name:<45} {m['accuracy']:>10.4f} {m['f1']:>10.4f}")

We can see that LoRA and the series adapter achieve very similar performance. However, given their architectures, LoRA is computationally more efficient, making it the more practical choice.

# 3 LoRA Rank Ablation Study
uilding on Model 5 (LoRA fine-tuning with rank=8), I conducted a systematic **rank
ablation study** by training five separate LoRA models with ranks of 2, 4, 8, 16,
and 32, while keeping all other hyperparameters identical. For each rank, I recorded
the test accuracy, weighted F1 score, and the number of trainable parameters, then
visualized the results across three dimensions:

- **Rank vs Accuracy** — does higher rank always lead to better performance?
- **Rank vs Trainable Parameters** — how does the parameter count scale with rank?
- **Parameters vs Accuracy (Efficiency Curve)** — at which rank does the
  performance-per-parameter ratio begin to diminish?

This experiment is interesting for two reasons:

**1. It directly answers a practical engineering question.**
When applying LoRA in practice, rank is the most critical hyperparameter to tune.
Too small a rank may underfit — the low-dimensional subspace cannot capture the
necessary domain shift from general English to financial language. Too large a rank
approaches full fine-tuning in parameter count and loses the efficiency advantage
of PEFT entirely. This ablation reveals the **sweet spot** where accuracy plateaus
and further increasing rank yields diminishing returns.

**2. It tests the low-rank hypothesis behind LoRA.**
LoRA is motivated by the hypothesis that the weight updates needed for fine-tuning
lie in a low-dimensional subspace — meaning a small rank should be sufficient to
capture all task-relevant adaptation. If this hypothesis holds for financial sentiment
classification, we would expect accuracy to plateau at a low rank (e.g., rank=4 or
rank=8) and not improve significantly beyond that. The ablation directly validates
or challenges this assumption on our specific dataset.

In [ ]:
RANKS_TO_TEST=[2,4,8,16,32]
rank_results={}

for RANK in RANKS_TO_TEST:
  print(f"\n{'='*50}")
  print(f"Testing LoRA Rank={RANK}")
  print(f"{'='*50}")

  model_rank=AutoModelForSequenceClassification.from_pretrained(
      model_name,
      num_labels=NUM_LABELS
  )
  model_rank.load_state_dict(best_state)

  for param in model_rank.parameters():
    param.requires_grad=False

  for layer in model_rank.distilbert.transformer.layer:
    layer.ffn.lin1=LoRALinear(layer.ffn.lin1,rank=RANK)
    layer.ffn.lin2=LoRALinear(layer.ffn.lin2,rank=RANK)

  model_rank.to(device)

  trainable_params=sum(p.numel() for p in model_rank.parameters() if p.requires_grad)

  optimizer_rank=torch.optim.AdamW(
      filter(lambda p: p.requires_grad, model_rank.parameters()),
      lr=1e-3,
      weight_decay=0.01
  )

  scheduler_rank=torch.optim.lr_scheduler.ReduceLROnPlateau(
      optimizer_rank,mode='max',patience=1,factor=0.5)

  criterion_rank=nn.CrossEntropyLoss()

  EPOCHS_RANK=5
  best_f1_rank=0.0
  best_state_rank=None

  for epoch in range(EPOCHS_RANK):
    model_rank.train()
    running_loss=0.0

    for batch in train_loader:
      input_ids=batch["input_ids"].to(device)
      attention_mask=batch["attention_mask"].to(device)
      labels=batch['label'].to(device)

      optimizer_rank.zero_grad()
      outputs=model_rank(input_ids=input_ids,attention_mask=attention_mask)
      loss=criterion_rank(outputs.logits,labels)
      loss.backward()
      optimizer_rank.step()
      running_loss+=loss.item()

    val_acc,val_f1,_,_=evaluate(model_rank,val_loader)
    scheduler_rank.step(val_f1)

    if val_f1>best_f1_rank:
      best_f1_rank=val_f1
      best_state_rank={k:v.clone() for k,v in model_rank.state_dict().items()}

    print(f"Epoch{epoch+1}/{EPOCHS_RANK} |"
      f"Loss: {running_loss/len(train_loader):.4f} |"
      f"Val F1: {val_f1:.4f}"
      +(" ← best" if val_f1 == best_f1_rank else""))

  model_rank.load_state_dict(best_state_rank)
  acc,f1,_,_=evaluate(model_rank,test_loader)
  rank_results[RANK]={
      "accuracy":acc,
      "f1":f1,
      "trainable_params":trainable_params
  }
  del model_rank
  torch.cuda.empty_cache()

In [ ]:
print("\n"+"="*60)
print("LoRA Rank Ablation Study - Results")
print("="*60)
print(f"{'Rank':>6} {'Trainable Params':>18} {'Accuracy':>10} {'F1':>10}")
print("-" * 60)
for rank, m in rank_results.items():
    print(f"{rank:>6} {m['trainable_params']:>18,} {m['accuracy']:>10.4f} {m['f1']:>10.4f}")

In [ ]:
ranks=list(rank_results.keys())
accs=[rank_results[r]["accuracy"] for r in ranks]
f1s=[rank_results[r]["f1"] for r in ranks]
param_counts=[rank_results[r]["trainable_params"] for r in ranks]

fig,axes=plt.subplots(1,3,figsize=(15,4))

axes[0].plot(ranks,accs,marker="o",color="#185FA5",linewidth=2)
axes[0].set_title("Accuracy vs Rank"); axes[0].set_xlabel("Rank")
axes[0].set_ylabel("Accuracy"); axes[0].grid(alpha=0.3)
axes[0].set_xticks(ranks)
for r,a in zip(ranks,accs):
  axes[0].annotate(f"{a:.4f}",(r,a),textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=9)

axes[1].bar(ranks, param_counts, color="#1D9E75", width=3)
axes[1].set_title("LoRA Rank vs Trainable Parameters")
axes[1].set_xlabel("LoRA Rank"); axes[1].set_ylabel("Trainable Parameters")
axes[1].set_xticks(ranks); axes[1].grid(alpha=0.3,axis="y")
for r, p in zip(ranks, param_counts):
    axes[1].text(r,p+1000,f"{p:,}",ha="center",fontsize=8)

axes[2].plot(param_counts, accs, marker="o", color="#D85A30", linewidth=2)
axes[2].set_title("Params vs Accuracy (Efficiency Curve)")
axes[2].set_xlabel("Trainable Parameters"); axes[2].set_ylabel("Accuracy")
axes[2].grid(alpha=0.3)
for r, p, a in zip(ranks, param_counts, accs):
    axes[2].annotate(f"rank={r}",(p,a),textcoords="offset points",
                     xytext=(5,-12), fontsize=8)

plt.suptitle("LoRA Rank Ablation Study", fontsize=13)
plt.tight_layout()
plt.show()

The ablation study reveals a clear and interpretable pattern across the five rank
configurations.

**Accuracy peaks at rank=8 and plateaus beyond that.**
Performance improves consistently from rank=2 (0.9448) to rank=8 (0.9558), but then
slightly declines at rank=16 (0.9536) and rank=32 (0.9536). This non-monotonic
behaviour suggests that beyond rank=8, the additional parameters do not provide
useful signal — instead, the larger adapter may be overfitting to the training set
given the relatively small size of the Financial PhraseBank dataset (~1,800 training
samples).

**The efficiency curve shows a sharp knee at rank=8.**
The right panel makes the trade-off explicit: moving from rank=2 to rank=8 yields a
+1.1% accuracy gain while adding only 276,480 parameters. Moving from rank=8 to
rank=32, however, adds over 1,100,000 parameters for a net accuracy *decrease* of
0.002. This confirms that rank=8 sits at the optimal point on the efficiency curve —
the largest accuracy gain per additional parameter.

**The low-rank hypothesis is partially supported.**
LoRA is motivated by the idea that fine-tuning updates lie in a low-dimensional
subspace. Our results partially support this: rank=8 (368,640 parameters) matches or
exceeds all higher-rank configurations, suggesting that the domain shift from general
English to financial sentiment can indeed be captured by a relatively low-dimensional
subspace. However, the fact that rank=2 and rank=4 underperform indicates that the
subspace dimensionality required is not trivially small — some minimum expressiveness
is needed to capture financial vocabulary and syntax.


## 4 In Context Learing
### What I am doing

In addition to the five fine-tuning based models, I explored **In-Context Learning
(ICL)** — a fundamentally different paradigm for adapting a language model to a new
task. Unlike all previous models which require gradient updates to adjust model
weights, ICL requires **zero training**: the model is given a small number of
labeled examples directly in the input prompt and is expected to infer the task
pattern from those examples alone.

Concretely, I constructed a **6-shot prompt** (2 examples per class) by selecting
representative sentences from the training set for each sentiment category. The
prompt follows this structure:

Classify the sentiment of financial sentences.
Sentiment can be: negative,neutral,or positive.

Sentence: <example 1>
Sentiment: negative

Sentence: <example 2>
Sentiment: neutral

Sentence: <example 3>
Sentiment: positive
...
Sentence: <test sentence>
Sentiment: ???

Since DistilBERT is an encoder-only model and is not designed for text generation,
I used **DistilGPT2** — a lightweight autoregressive language model — for this
experiment. The prediction is made by scoring each candidate label (negative,
neutral, positive) using the model's language modeling loss: the label with the
lowest perplexity when appended to the prompt is selected as the prediction.

I also ran a **zero-shot** baseline (no examples in the prompt) to isolate the
specific contribution of the few-shot examples.

### Why this is interesting

**1. It represents a completely different learning paradigm.**
All five required models learn from data by updating weights through backpropagation.
ICL, by contrast, achieves task adaptation purely through the structure of the input
— no parameters are modified at inference time. Comparing ICL against fine-tuning
directly quantifies the value of gradient-based training: how much does actually
updating the model weights improve over simply showing it a few examples?

**2. It tests the limits of zero-parameter adaptation.**
A practical question in NLP is: when labeled data is scarce, is it better to
fine-tune a small model or to use ICL with a larger pretrained model? Our experiment
provides a concrete data point — if few-shot ICL with DistilGPT2 approaches the
accuracy of Model 2 (head-only fine-tuning), it suggests that ICL could be a viable
alternative when fine-tuning infrastructure is unavailable.

**3. The zero-shot vs few-shot comparison isolates the value of examples.**
By running both zero-shot (no examples) and 6-shot (2 per class) conditions, we can
directly measure how much each additional example contributes to performance. If the
gap between zero-shot and few-shot is large, it confirms that the model is genuinely
performing in-context learning rather than relying on prior knowledge about financial
sentiment embedded during pretraining.



In [ ]:
from transformers import DistilBertForMaskedLM, DistilBertTokenizerFast

MLM_MODEL_NAME = "distilbert-base-uncased"
mlm_tokenizer  = DistilBertTokenizerFast.from_pretrained(MLM_MODEL_NAME)
mlm_model      = DistilBertForMaskedLM.from_pretrained(MLM_MODEL_NAME)
mlm_model.to(device)
mlm_model.eval()


In [ ]:
from collections import defaultdict
import random

random.seed(SEED)

examples_by_class = defaultdict(list)
for item in train_dataset:
    examples_by_class[item["label"]].append(item["sentence"])

N_SHOTS = 2   # 每类 2 个示例，共 6 个
few_shot_examples = []
for label_id, label_name in enumerate(label_names):
    chosen = random.sample(examples_by_class[label_id], N_SHOTS)
    for sentence in chosen:
        few_shot_examples.append((sentence, label_name))

print("Few-shot example：")
for sent, lab in few_shot_examples:
    print(f"  [{lab}] {sent[:80]}...")

In [ ]:
def build_mlm_prompt(test_sentence,examples):
  prompt = ""
  for sentence, label in examples:
      prompt += f"sentence : {sentence}\n"
      prompt += f"sentiment : {label}\n\n"

  prompt += f"sentence : {test_sentence}\n"
  prompt += f"sentiment : [MASK]"
  return prompt

sample_prompt = build_mlm_prompt(test_dataset[0]["sentence"], few_shot_examples)
print("=" * 60)
print("Prompt example：")
print("=" * 60)
print(sample_prompt)



In [ ]:
def icl_predict_bert(test_sentence,examples,tokenizer,model,device):
  label_names = ["negative", "neutral", "positive"]

  prompt=build_mlm_prompt(test_sentence, examples)
  inputs=tokenizer(
      prompt,
      return_tensors="pt",
      truncation=True,
      max_length= 10000,
  )
  input_ids=inputs["input_ids"].to(device)
  attention_mask=inputs["attention_mask"].to(device)


  mask_token_id=tokenizer.mask_token_id
  mask_positions=(input_ids==mask_token_id).nonzero(as_tuple=True)

  if len(mask_positions[1]) == 0:
      print("No mask token in prompt!")
      return random.randint(0, 2), [0.0, 0.0, 0.0]

  mask_pos = mask_positions[1][0]

  with torch.no_grad():
      outputs=model(input_ids=input_ids, attention_mask=attention_mask)
      mask_logits=outputs.logits[0, mask_pos, :]


  scores = []
  for label in label_names:
      tokens=tokenizer.tokenize(label)
      token_id=tokenizer.convert_tokens_to_ids(tokens[0])
      scores.append(mask_logits[token_id].item())

  pred_idx=int(np.argmax(scores))
  return pred_idx, scores


In [ ]:
N_TEST_SAMPLES=200
test_subset=test_dataset.select(range(N_TEST_SAMPLES))

print("evaluate Zero-shot ICL...")
preds_0shot, labels_0shot = [], []
for i, item in enumerate(test_subset):
    pred, _ = icl_predict_bert(
      item["sentence"],[],mlm_tokenizer,mlm_model,device
    )
    preds_0shot.append(pred)
    labels_0shot.append(item["label"])
    if (i + 1) % 50 == 0:
        print(f"{i+1}/{N_TEST_SAMPLES} done...")

acc_0shot=accuracy_score(labels_0shot, preds_0shot)
f1_0shot=f1_score(labels_0shot, preds_0shot, average="weighted")
print(f"Zero-shot → Accuracy: {acc_0shot:.4f} | F1: {f1_0shot:.4f}\n")

print("evaluate Few-shot ICL (6-shot)...")
preds_6shot, labels_6shot = [], []
for i, item in enumerate(test_subset):
    pred,_=icl_predict_bert(
        item["sentence"],few_shot_examples,mlm_tokenizer,mlm_model,device
    )
    preds_6shot.append(pred)
    labels_6shot.append(item["label"])
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_TEST_SAMPLES} done...")

acc_6shot=accuracy_score(labels_6shot, preds_6shot)
f1_6shot=f1_score(labels_6shot, preds_6shot, average="weighted")
print(f" Few-shot  → Accuracy: {acc_6shot:.4f} | F1: {f1_6shot:.4f}")

In [ ]:
print("\n" + "=" * 50)
print("  Zero-shot ICL (DistilBERT MLM)")
print("=" * 50)
print(classification_report(labels_0shot, preds_0shot, target_names=label_names))

print("=" * 50)
print("  Few-shot ICL (DistilBERT MLM, 6-shot)")
print("=" * 50)
print(classification_report(labels_6shot, preds_6shot, target_names=label_names))

### Results and Analysis

#### Overall Performance

The results reveal a surprising and counterintuitive finding: **Few-shot ICL
(0.24) performs significantly worse than Zero-shot ICL (0.38)**, which itself
barely exceeds random chance (0.33). This outcome is worth examining carefully,
as it exposes fundamental limitations of applying ICL to encoder-only models.



#### Why Does Few-shot Hurt? A Fundamental Limitation

This result exposes a core incompatibility between ICL and MLM-based models:

**1. DistilBERT was not designed for ICL.**
ICL was developed for autoregressive models (GPT-style), where the model
is trained to predict the next token given all previous context. DistilBERT,
by contrast, is trained to predict randomly masked tokens in isolation. When
presented with a long few-shot prompt, DistilBERT does not "read" the examples
sequentially and infer a pattern — it simply uses bidirectional attention to
fill in [MASK] based on surrounding context, which is a fundamentally different
operation.

**2. The prompt length interferes with the [MASK] prediction.**
As the prompt grows longer with more examples, the [MASK] token receives
attention from a much larger and more complex context. This disrupts the
logit distribution at the [MASK] position in unpredictable ways, causing
the model to fixate on whichever candidate token happens to have the
highest logit — in this case, "negative" — regardless of the actual
sentence content.

**3. The scoring mechanism is suboptimal for MLM.**
Comparing only the logits of three specific tokens (negative/neutral/positive)
out of a vocabulary of 30,522 is a coarse approximation. The relative logit
values of these three tokens are sensitive to the prompt structure and may
not reliably reflect the model's "understanding" of the sentiment.


